[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SEU_USUARIO/gr-notebooks/blob/main/aula-big-data/aula-01-conceitos-basicos-de-big-data.ipynb)

# Conceitos Básicos de Big Data

Você certamente já ouviu — ou usou — o termo "Big Data" em alguma reunião, vaga de emprego ou matéria de tecnologia. É um dos termos mais citados e, ao mesmo tempo, mais mal definidos do mercado de dados: para uns é sinônimo de "muitos dados", para outros é qualquer coisa que envolva Spark ou Hadoop, e para outros ainda é só um jargão de marketing.

O objetivo desta aula é sair do modismo e chegar a critérios objetivos: o que caracteriza, de fato, um problema de Big Data, quando vale a pena adotar ferramentas distribuídas em vez de uma solução tradicional, e quais são as peças que compõem o ecossistema de ferramentas dessa área — armazenamento distribuído, processamento distribuído, arquitetura em camadas e orquestração.

Para tornar os conceitos concretos, vamos usar ao longo da aula um conjunto de dados real de funcionários de uma empresa, organizado em arquivos particionados por mês de extração. O foco aqui está no vocabulário e nos conceitos fundamentais, não na implementação de nenhuma ferramenta específica.

## Preparando o Ambiente

Vamos usar um conjunto de dados de exemplo com informações de funcionários de uma empresa fictícia, já organizado em arquivos CSV particionados por mês de extração (pasta `data/landing/funcionarios/`, com subpastas `dt=AAAA-MM-DD`). Essa organização em partições é, inclusive, um dos primeiros conceitos de Big Data que vamos ver na prática ainda nesta aula.

> 📝 **Nota:** Se você estiver rodando este notebook localmente a partir do repositório, os caminhos relativos abaixo já funcionam. Se estiver no Google Colab, descomente as linhas da célula abaixo para clonar o repositório antes de continuar.

In [ ]:
# se estiver executando no Google Colab, descomente as linhas abaixo para clonar o repositório com os dados desta aula
# !git clone https://github.com/SEU_USUARIO/gr-notebooks.git
# %cd gr-notebooks/aula-big-data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# biblioteca padrão do Python usada para navegar pelos arquivos da landing zone
import glob

## O que é Big Data, afinal?

"Big Data" é um daqueles termos que viraram modismo e, por isso mesmo, perderam um pouco do significado. Então vamos direto ao ponto: **Big Data não é sobre o tamanho do dado, é sobre o limite das suas ferramentas.**

Durante anos, a forma padrão de trabalhar com dados foi: um arquivo (ou uma tabela de banco relacional), processado por um único programa, rodando em uma única máquina, com o dado inteiro cabendo na memória RAM disponível. O Pandas, por exemplo, foi feito exatamente para esse cenário.

O problema é que, em algum momento, um (ou mais) destes três limites é ultrapassado:

- O **volume** de dados fica grande demais para caber na memória de uma única máquina.
- A **velocidade** com que novos dados chegam é maior do que a capacidade de processá-los em tempo hábil.
- A **variedade** de formatos e fontes de dados é grande demais para caber num modelo único de tabela.

Quando isso acontece, é preciso uma abordagem diferente: distribuir o armazenamento e o processamento dos dados entre várias máquinas trabalhando em conjunto. É esse conjunto de técnicas, arquiteturas e ferramentas que chamamos de Big Data.

> 💡 **Dica:** Uma pergunta prática que vale sempre fazer antes de sair usando ferramentas de Big Data: "os meus dados cabem confortavelmente na memória da minha máquina?" Se a resposta é sim, você provavelmente não precisa de Big Data — e não tem problema nenhum nisso.

### Um problema mais antigo do que o termo

O termo "Big Data" só se popularizou no vocabulário do mercado a partir dos anos 2000, mas o problema que ele descreve é bem mais antigo. Um bom exemplo é a reportagem [*"It's sink or swim as a tidal wave of data approaches"*](https://www.nature.com/articles/21044), publicada pela revista científica **Nature** em 1999 (Reichhardt, T. *Nature* 399, 517–518, 1999. doi:10.1038/21044) — dez anos antes do termo "Big Data" entrar no vocabulário popular.

A reportagem descrevia exatamente o problema que vimos acima: áreas como genômica e astronomia estavam acumulando dados em um ritmo muito maior do que a capacidade de analisá-los, e isso exigiria investimento pesado em infraestrutura computacional, novos métodos estatísticos e práticas de compartilhamento aberto de dados entre laboratórios. Ou seja: o texto usa expressões como "*tidal wave of data*" (uma maré de dados) em vez de "Big Data", mas o diagnóstico — ferramentas e processos que não acompanham o crescimento dos dados — é exatamente o mesmo que discutimos hoje.

## Os 5 V's do Big Data

Uma forma clássica (e ainda muito útil) de caracterizar um problema de Big Data é através dos **V's**. E, ao contrário de boa parte do vocabulário de Big Data, esse conjunto de critérios tem uma origem bem documentada.

Em fevereiro de 2001, o analista **Doug Laney**, então na consultoria META Group (mais tarde adquirida pela Gartner), publicou uma nota de pesquisa intitulada *"3D Data Management: Controlling Data Volume, Velocity, and Variety"*. Anos depois, ele reconstituiu a origem do conceito em detalhe no artigo [*"Deja VVVu: Gartner's Original 'Volume-Velocity-Variety' Definition of Big Data"*](https://community.aiim.org/blogs/doug-laney/2012/08/25/deja-vvvu-gartners-original-volume-velocity-variety-definition-of-big-data) (Laney, D., 2012): segundo ele, no fim dos anos 1990 os clientes da consultoria estavam cada vez mais sobrecarregados por seus próprios dados — não só pelo volume crescente, mas pela velocidade com que o comércio eletrônico passou a gerar dados, e pela variedade de fontes e formatos que se multiplicaram com a onda de sistemas ERP pós-Y2K. Os três V's originais — **Volume, Velocidade e Variedade** — nasceram para descrever esses três eixos de dificuldade, não o "tamanho dos dados" isoladamente.

Nos anos seguintes, outras empresas e autores da indústria (como a IBM) incorporaram mais dois V's ao modelo — **Veracidade** e **Valor** — formando o conjunto de 5 V's que se tornou padrão de mercado. Vamos ver cada um deles com exemplos práticos, inclusive usando o conjunto de dados de funcionários que vamos explorar nesta aula:

- **Volume**: a quantidade total de dados. No conjunto de dados usado nesta aula, cada extração mensal traz algumas centenas de funcionários — mas em uma empresa real, com centenas de milhares de funcionários e anos de histórico, esse volume cresce rapidamente.
- **Velocidade**: a frequência com que novos dados chegam. As extrações que vamos analisar são mensais (`dt=2025-09-01`, `dt=2025-10-01`, ...), um cenário de *batch*. Em outros contextos, os dados poderiam chegar em tempo real, conforme os eventos acontecem.
- **Variedade**: os diferentes formatos e estruturas de dados. É comum um mesmo pipeline lidar com arquivos de funcionários, departamentos, cargos e eventos, cada um com seu próprio esquema — e, na prática, ainda teríamos PDFs de contratos, planilhas, dados de sistemas de ponto, etc.
- **Veracidade**: a confiabilidade e qualidade dos dados. Nomes duplicados, e-mails inconsistentes, datas que não batem entre sistemas diferentes — tudo isso precisa ser tratado antes que os dados sejam confiáveis para análise.
- **Valor**: no fim das contas, todo esse esforço só se justifica se os dados gerarem valor real para alguém tomar uma decisão. Sem isso, nenhum investimento em arquitetura de dados se sustenta.

> 📝 **Nota:** Você também vai ver referências a um 6º ou 7º V (Variabilidade, Visualização) dependendo da fonte. Os 5 acima são os mais consolidados e já são suficientes para guiar boa parte das decisões de arquitetura de dados.

## Colocando a Mão na Massa: Medindo o Volume dos Nossos Dados

Antes de falar de ferramentas, vamos sentir na prática o que significa ter dados particionados por data de extração — um padrão comum em pipelines de ingestão de sistemas de RH/HCM, onde cada extração é uma "fotografia" completa da base naquele momento.

Vamos carregar todas as partições mensais da tabela `funcionarios` em um único DataFrame do pandas.

In [ ]:
# listando e lendo todas as partições mensais da tabela de funcionários na landing zone
caminhos_particoes = sorted(glob.glob('../data/landing/funcionarios/dt=*/funcionarios.csv'))
df_funcionarios = pd.concat([pd.read_csv(c) for c in caminhos_particoes], ignore_index=True)

df_funcionarios.head(3)

Repare que a coluna `data_extracao`, presente no próprio arquivo, já identifica de qual partição mensal cada linha veio — não precisamos interpretar o nome das pastas para isso. Podemos usar um `groupby` do pandas para contar quantas linhas cada extração mensal trouxe.

In [ ]:
# contando o número de linhas de cada partição mensal, usando a própria coluna data_extracao
linhas_por_particao = df_funcionarios.groupby('data_extracao').size()
linhas_por_particao

A base de funcionários cresce mês a mês: são mais admissões do que desligamentos, então cada nova extração traz mais linhas do que a anterior. Vamos visualizar esse crescimento.

In [ ]:
# configurações visuais padrão do gráfico
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.spines.top"] = False

# plotando a evolução do número de linhas por partição mensal direto a partir da Series do pandas
linhas_por_particao.plot(marker='o')

# adicionando título e rótulos dos eixos
plt.title('Crescimento da Base de Funcionários por Partição Mensal')
plt.xlabel('Mês de Extração')
plt.ylabel('Número de Linhas na Partição')

# rotacionando os rótulos do eixo X para melhor leitura
plt.xticks(rotation=45)

# exibe o gráfico
plt.show()

Agora um ponto importante: cada partição mensal é uma **extração completa** da base de funcionários, não só os funcionários novos daquele mês. Isso significa que, ao concatenarmos todas as partições (como já fizemos ao montar `df_funcionarios`), o mesmo funcionário aparece repetido em várias delas. Vamos comprovar isso comparando o total de linhas com o total de funcionários únicos:

In [ ]:
# comparando o total de linhas com o total de funcionários únicos
print(f'Total de linhas somando todas as partições: {len(df_funcionarios):,}')
print(f'Total de funcionários únicos: {df_funcionarios["funcionario_id"].nunique():,}')

O número de linhas é bem maior do que o número de funcionários únicos — a maior parte dos dados que "somamos" é, na verdade, repetição do mesmo funcionário aparecendo em extração após extração. Esse tipo de duplicidade entre extrações completas é extremamente comum em pipelines reais, e normalmente é resolvido com técnicas de deduplicação e de controle de histórico de dados (uma técnica conhecida como *Slowly Changing Dimensions*, ou SCD).

Agora imagine esse mesmo cenário em uma empresa com 500 mil funcionários e 10 anos de histórico mensal: seriam dezenas de milhões de linhas, a maior parte redundante, e nenhuma chance de abrir isso tudo de uma vez na memória de uma única máquina com Pandas.

> ⚠️ **Atenção:** No nosso exemplo, os arquivos são pequenos o suficiente para o Pandas processar sem problema — e é exatamente por isso que ele é uma ótima ferramenta para prototipar e entender o formato dos dados antes de migrar a lógica para uma ferramenta distribuída, como veremos a seguir.

## Quando o Pandas (e o Excel) Deixam de Ser Suficientes

O Pandas carrega o DataFrame inteiro na memória RAM da sua máquina. Isso funciona muito bem até os dados ocuparem alguns gigabytes — a partir daí, dependendo da memória disponível, o processo trava, fica lento ou simplesmente estoura (o famoso `MemoryError`).

Existem basicamente três sinais de que chegou a hora de considerar ferramentas de Big Data:

1. **Os dados não cabem na memória** de uma única máquina, mesmo em máquinas robustas.
2. **O tempo de processamento é inviável**, mesmo que os dados caibam na memória (por exemplo, um `join` que levaria horas em uma única máquina).
3. **Os dados estão espalhados** em múltiplas fontes e sistemas, e precisam ser combinados de forma consistente e repetível.

> 💡 **Dica:** Ferramentas de Big Data como o Spark também funcionam muito bem com dados pequenos! A diferença é que elas continuam funcionando quando os dados crescem, enquanto o Pandas, em algum momento, simplesmente para de funcionar.

## O Ecossistema de Big Data

Para lidar com volume, velocidade e variedade em escala, o ecossistema de Big Data se organiza, de forma simplificada, em quatro grandes peças.

### Armazenamento Distribuído

Em vez de guardar os arquivos no disco de uma única máquina, os dados ficam distribuídos em um sistema de armazenamento capaz de crescer horizontalmente (adicionando mais discos/máquinas) em vez de verticalmente (trocando por um disco maior).

O projeto que popularizou essa ideia foi o **Google File System (GFS)**, descrito no paper [*"The Google File System"*](https://research.google/pubs/the-google-file-system/) (Ghemawat, S., Gobioff, H., & Leung, S.-T. *Proceedings of the 19th ACM Symposium on Operating Systems Principles* — SOSP '03, 2003). A ideia central do GFS é reveladora de como se pensa em escala: em vez de tratar falhas de hardware como exceção, o sistema assume que **componentes vão falhar o tempo todo** — os autores explicam que, com milhares de máquinas construídas a partir de peças baratas, algumas simplesmente não vão funcionar em um dado momento, e o sistema precisa de monitoramento constante, detecção de erros e recuperação automática como parte do próprio design.

Na prática, o GFS funciona assim: cada arquivo é dividido em blocos fixos de **64 MB**, chamados de *chunks*, e cada chunk é replicado em **três máquinas diferentes** por padrão, para tolerar falhas. Um único servidor "mestre" guarda os metadados (em qual máquina está cada chunk), enquanto dezenas ou centenas de servidores de armazenamento (*chunkservers*) guardam os dados propriamente ditos. O **Amazon S3** é um exemplo real e amplamente usado desse mesmo tipo de armazenamento distribuído, cumprindo em escala o mesmo papel que a pasta local `data/landing/` está cumprindo neste notebook: um repositório central para os arquivos brutos antes de qualquer processamento.

### Processamento Distribuído

Ter os dados espalhados por várias máquinas não adianta nada se o processamento continuar preso a uma única máquina. O paper que resolveu esse problema em escala industrial foi o [*"MapReduce: Simplified Data Processing on Large Clusters"*](https://research.google/pubs/mapreduce-simplified-data-processing-on-large-clusters/) (Dean, J., & Ghemawat, S. *OSDI'04: 6th Symposium on Operating Systems Design and Implementation*, 2004).

A ideia do MapReduce é simples de enunciar: o programador escreve apenas duas funções — uma função `map`, que processa cada registro de entrada e gera pares chave/valor intermediários, e uma função `reduce`, que combina todos os valores associados à mesma chave. O sistema cuida sozinho de tudo o mais: dividir os dados, distribuir o trabalho entre as máquinas, lidar com falhas (reexecutando automaticamente as tarefas que falharem em outra máquina) e coordenar a comunicação entre elas. Segundo os próprios autores, uma computação MapReduce típica no Google processava "muitos terabytes de dados em milhares de máquinas", com mais de mil desses jobs rodando todos os dias nos clusters da empresa — em 2004. Essa ideia foi depois reimplementada como projeto de código aberto pelo Hadoop MapReduce.

O sucessor natural do MapReduce é o **Apache Spark**, apresentado no paper [*"Resilient Distributed Datasets: A Fault-Tolerant Abstraction for In-Memory Cluster Computing"*](https://www.usenix.org/conference/nsdi12/technical-sessions/presentation/zaharia) (Zaharia, M. et al. *9th USENIX Symposium on Networked Systems Design and Implementation* — NSDI '12, 2012, vencedor do Best Paper Award). A limitação do MapReduce clássico é que, entre uma etapa e outra, os resultados intermediários precisam ser gravados em disco — o que é lento para algoritmos iterativos (como os de machine learning) que reprocessam os mesmos dados várias vezes. O Spark resolve isso com as *Resilient Distributed Datasets* (RDDs): estruturas de dados que ficam **em memória** entre as operações, o que os próprios autores relatam melhorar o desempenho "em uma ordem de grandeza" (ou seja, cerca de 10x) para cargas de trabalho iterativas — e ainda assim mantém tolerância a falhas, reconstruindo apenas a parte perdida a partir do histórico de transformações (a *lineage*), em vez de precisar replicar fisicamente os dados como o GFS faz.

### Arquitetura Lakehouse e Camadas Medalhão

Um **Data Lakehouse** combina a flexibilidade e o baixo custo de um Data Lake (armazenar qualquer tipo de arquivo, sem estrutura rígida) com as garantias de qualidade e organização de um Data Warehouse tradicional. O termo e a arquitetura foram formalizados no paper [*"Lakehouse: A New Generation of Open Platforms that Unify Data Warehousing and Advanced Analytics"*](http://cidrdb.org/cidr2021/papers/cidr2021_paper17.pdf) (Armbrust, M., Ghodsi, A., Xin, R., & Zaharia, M. *11th Conference on Innovative Data Systems Research* — CIDR '21, 2021).

Os autores argumentam que a arquitetura clássica de "data lake + data warehouse" (dados brutos no lake, depois copiados via ETL para um warehouse antes de qualquer análise) sofre de quatro problemas recorrentes: **confiabilidade** (manter as duas cópias sincronizadas é caro e sujeito a erros), **desatualização dos dados** (o warehouse fica sempre um passo atrás do lake), **suporte limitado a analytics avançado** (ferramentas de machine learning não leem bem os formatos de warehouse) e **custo total de propriedade** (pagar por dois sistemas). Como evidência do problema de desatualização, os autores citam uma pesquisa segundo a qual 86% dos analistas usam dados desatualizados e 62% relatam esperar repetidamente por recursos de engenharia todo mês. A proposta do Lakehouse é acabar com essa duplicação: usar formatos abertos de acesso direto (como o **Apache Parquet**) diretamente sobre o armazenamento de baixo custo do data lake, adicionando uma camada de gerenciamento (metadados, transações, controle de qualidade) por cima.

A forma mais comum de organizar um Lakehouse na prática é a **arquitetura medalhão**, dividida em três camadas:

- **Bronze**: os dados chegam crus, exatamente como foram extraídos da origem, apenas com metadados de auditoria (arquivo de origem, timestamp de ingestão).
- **Silver**: os dados são limpos, deduplicados, tipados corretamente e enriquecidos.
- **Gold**: os dados são modelados para consumo analítico, geralmente em um modelo dimensional (fato + dimensões), pronto para alimentar dashboards e relatórios.

### Orquestração

Um pipeline de Big Data raramente roda "uma vez só". Ele precisa ser executado periodicamente (todo dia, toda hora, a cada nova partição), respeitando dependências entre etapas (uma camada de dados limpos só pode ser gerada depois que os dados crus terminaram de chegar, por exemplo) e lidando com falhas.

O **Apache Airflow** é hoje a ferramenta de orquestração mais usada para esse tipo de problema. Ele nasceu dentro do Airbnb, e sua criação foi anunciada publicamente no artigo [*"Airflow: a workflow management platform"*](https://medium.com/airbnb-engineering/airflow-a-workflow-management-platform-46318b977fd8) (Beauchemin, M. *Airbnb Engineering & Data Science*, 2 de junho de 2015), quando a empresa abriu o código da ferramenta que já usava internamente para lidar com a quantidade crescente de pipelines de dados. A ideia central do Airflow é representar cada pipeline como um **DAG** (*Directed Acyclic Graph*, ou grafo acíclico dirigido): cada etapa do pipeline é um nó do grafo, e as setas entre os nós representam as dependências entre elas — garantindo, por exemplo, que a camada Silver só comece depois que a Bronze tiver terminado com sucesso.

## Modelagem de Dados: de Tabelas Normalizadas ao Modelo Estrela

Quando descrevemos a camada Gold como um "modelo dimensional (fato + dimensões)", passamos rápido por um assunto que merece mais atenção: como, exatamente, se organiza um conjunto de tabelas para consumo analítico? Essa pergunta é respondida por uma área da engenharia de dados chamada **modelagem dimensional**, que tem tanta teoria — e tanta história — quanto o próprio Big Data.

### OLTP vs. OLAP: Dois Mundos com Objetivos Diferentes

A maioria dos sistemas de origem de um pipeline de dados — o sistema que registra uma venda, um cadastro, uma transação — é o que chamamos de **OLTP** (*Online Transaction Processing*). Esses sistemas são otimizados para muitas transações pequenas, rápidas e simultâneas: inserir um registro, atualizar um campo, sem deixar o banco de dados em um estado inconsistente mesmo com múltiplos usuários escrevendo ao mesmo tempo.

A base teórica por trás da maioria desses sistemas é o **modelo relacional**, formalizado no paper [*"A Relational Model of Data for Large Shared Data Banks"*](https://github.com/tpn/pdfs/blob/master/A%20Relational%20Model%20of%20Data%20for%20Large%20Shared%20Data%20Banks%20-%20E.F.%20Codd%20(1970).pdf) (Codd, E. F. *Communications of the ACM*, 13(6), 377–387, 1970). Codd propôs que os dados fossem representados como relações (tabelas) descritas apenas por sua estrutura lógica, sem expor a forma como estão fisicamente armazenados — e, a partir daí, consolidou-se a prática de **normalização**: organizar os dados para eliminar redundância, de forma que cada informação exista em um único lugar. Isso é ótimo para escrita (atualizar um endereço em um único lugar, por exemplo) mas caro para leitura analítica, porque uma pergunta simples ("quanto vendemos por região no último trimestre?") pode exigir juntar dezenas de tabelas normalizadas.

É aí que entra o **OLAP** (*Online Analytical Processing*): sistemas otimizados para poucas consultas, porém pesadas, sobre grandes volumes de dados históricos — exatamente o papel da camada Gold do nosso Lakehouse. Para tornar essas consultas rápidas e compreensíveis para humanos, os dados de um sistema OLAP costumam ser **desnormalizados de propósito**, aceitando alguma redundância controlada em troca de menos `joins` e consultas mais diretas.

### O Modelo Dimensional: Fatos e Dimensões

A técnica mais usada para desnormalizar dados analíticos é a **modelagem dimensional**, popularizada por Ralph Kimball. No artigo [*"Fact Tables and Dimension Tables"*](https://www.kimballgroup.com/2003/01/fact-tables-and-dimension-tables/) (Kimball, R. *Kimball Group*, 2003), ele resume a ideia central: todo processo de negócio pode ser descrito por **medições numéricas** (os fatos) cercadas de **contexto descritivo** (as dimensões).

- Uma **tabela fato** guarda as medições de um processo de negócio (quantidade, valor, duração) junto com chaves estrangeiras que apontam para cada dimensão relevante.
- Uma **tabela dimensão** guarda o contexto descritivo em torno de cada fato — o "quem", "o quê", "onde" e "quando" de cada medição — geralmente com poucas linhas e muitas colunas de texto.

Imagine, por exemplo, uma tabela fato de vendas: cada linha registraria uma venda (quantidade, valor), com chaves estrangeiras para as dimensões de produto, cliente, loja e data. Quando a tabela fato fica no centro e as dimensões ao redor, ligadas diretamente a ela, o desenho lembra uma estrela — daí o nome **star schema** (esquema estrela). Quando as próprias dimensões são normalizadas em subtabelas, o desenho se ramifica ainda mais e passa a ser chamado de **snowflake schema** (esquema floco de neve).

Antes de desenhar qualquer tabela fato, Kimball recomenda declarar explicitamente o seu **grão** (*grain*): o que, exatamente, representa uma linha da tabela. No artigo [*"Keep to the Grain in Dimensional Modeling"*](https://www.kimballgroup.com/2007/07/keep-to-the-grain-in-dimensional-modeling/) (Kimball, R. *Kimball Group*, 2007), ele defende que o grão deve sempre começar no nível mais atômico possível, determinado pela realidade física de como o dado é coletado na origem — e não pelas perguntas de negócio que imaginamos responder hoje, já que essas perguntas mudam com o tempo.

> 📝 **Nota:** Nem todo mundo modela dados analíticos da mesma forma que Kimball. Bill Inmon, autor de *"Building the Data Warehouse"* (Wiley, início dos anos 1990), defende uma abordagem "de cima para baixo": construir primeiro um modelo corporativo único e normalizado, e só depois derivar visões dimensionais para áreas específicas — o oposto da abordagem "de baixo para cima" de Kimball, que modela cada processo de negócio direto em fatos e dimensões. Essa discussão ficou conhecida como o debate **Inmon vs. Kimball**, e arquiteturas modernas costumam misturar as duas ideias: uma camada mais normalizada e limpa (nossa Silver) alimentando um modelo dimensional (nossa Gold).

### Os Três Tipos de Tabela Fato

Nem toda tabela fato guarda o mesmo tipo de medição. Na literatura de modelagem dimensional, os fatos costumam se encaixar em um destes três tipos:

- **Fato de transação** ([*Transaction Fact Tables*](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/kimball-techniques/dimensional-modeling-techniques/transaction-fact-table/), Kimball Group): uma linha para cada evento de negócio, no grão mais atômico possível — uma venda, um clique, um evento de RH (admissão, promoção, desligamento). É o tipo mais comum, e o que vamos usar na demonstração desta aula.
- **Fato de snapshot periódico** ([*Periodic Snapshot Fact Tables*](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/kimball-techniques/dimensional-modeling-techniques/periodic-snapshot-fact-table/), Kimball Group): uma linha para cada período de tempo (dia, mês), resumindo o estado de algo naquele momento — por exemplo, o headcount de cada departamento ao final de cada mês. Diferente do fato de transação, aqui toda entidade aparece em toda linha do período, mesmo sem nenhum evento novo, o que torna a tabela previsivelmente densa.
- **Fato de snapshot cumulativo** ([*Accumulating Snapshot Fact Tables*](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/kimball-techniques/dimensional-modeling-techniques/accumulating-snapshot-fact-table/), Kimball Group): uma linha por processo com início e fim bem definidos (um pedido, um chamado de suporte), atualizada à medida que o processo avança por etapas previsíveis — cada etapa ganha sua própria coluna de data, permitindo medir o tempo entre marcos do processo.

> 💡 **Dica:** as extrações mensais de `funcionarios` e `departamentos` que usamos na seção prática — uma "fotografia" completa da base a cada mês — têm a cara de um fato de snapshot periódico. A diferença é que, na Gold, usamos essas fotografias como matéria-prima para *dimensões* (o estado de cada entidade), não como fato — o que mostra que a fronteira entre fato e dimensão depende da pergunta de negócio que você quer responder, não só do formato do dado de origem.

### Além da Estrela Básica: Outros Tipos de Dimensão

O par fato/dimensão que vimos até aqui é só o ponto de partida. Kimball descreve outras variações que aparecem o tempo todo em modelos reais:

- **Dimensão degenerada** ([*Degenerate Dimensions*](https://www.kimballgroup.com/2003/06/design-tip-46-another-look-at-degenerate-dimensions/), Design Tip #46, Kimball Group, 2003): um identificador do processo de negócio — como um número de pedido ou de nota fiscal — que fica guardado direto na tabela fato, sem uma tabela dimensão própria, porque não carrega nenhum atributo descritivo além de si mesmo.
- **Dimensão conformada** ([*Conformed Dimensions*](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/kimball-techniques/dimensional-modeling-techniques/conformed-dimension/), Kimball Group): a mesma tabela dimensão, com a mesma definição, reutilizada por várias tabelas fato diferentes — por exemplo, uma `dim_departamento` única usada tanto por uma fato de eventos de RH quanto por uma futura fato de folha de pagamento, garantindo que "departamento" signifique exatamente a mesma coisa nas duas análises.
- **Dimensão banana (*junk dimension*)** ([*Junk Dimensions*](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/kimball-techniques/dimensional-modeling-techniques/junk-dimension/), Kimball Group): agrupa vários indicadores e sinalizadores de baixa cardinalidade — sem relação direta entre si — em uma única tabela dimensão, evitando poluir a fato com uma chave estrangeira para cada mini-atributo.
- **Dimensão com múltiplos papéis (*role-playing dimension*)** ([*Role-Playing Dimensions*](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/kimball-techniques/dimensional-modeling-techniques/role-playing-dimension/), Kimball Group): a mesma tabela física de dimensão é referenciada mais de uma vez pela mesma fato, com significados diferentes — o exemplo clássico é uma `dim_data` usada como "data do pedido" e, na mesma linha, como "data de entrega".

> 📝 **Nota:** todas essas variações continuam obedecendo à mesma regra de ouro do modelo dimensional: fatos guardam medições numéricas, dimensões guardam contexto descritivo. Elas só resolvem detalhes específicos de como organizar esse contexto — não mudam o desenho básico da estrela.

### Slowly Changing Dimensions: Modelando Mudanças ao Longo do Tempo

Lembra do problema que vimos lá atrás, na seção prática, quando o mesmo funcionário aparecia repetido em cada extração mensal? Isso é exatamente o tipo de situação que a técnica de ***Slowly Changing Dimensions*** (SCD, ou "dimensões de mudança lenta") existe para resolver: como representar, em uma tabela dimensão, um atributo que muda ao longo do tempo — um cargo, um endereço, uma categoria de produto — sem perder o histórico nem duplicar dados sem controle?

Nos artigos [*"Slowly Changing Dimensions"*](https://www.kimballgroup.com/2008/08/slowly-changing-dimensions/) e [*"Slowly Changing Dimensions, Part 2"*](https://www.kimballgroup.com/2008/09/slowly-changing-dimensions-part-2/) (Kimball, R. *Kimball Group*, 2008), Kimball descreve as duas técnicas mais usadas na prática:

- **Tipo 1 (sobrescrever)**: o valor antigo é simplesmente substituído pelo novo. É simples e ocupa pouco espaço, mas destrói o histórico — qualquer análise que dependesse do valor anterior deixa de ser possível.
- **Tipo 2 (adicionar uma linha)**: em vez de sobrescrever, uma **nova linha** é inserida na dimensão com uma nova chave substituta (*surrogate key*), preservando a linha antiga intacta. Kimball descreve cinco campos de controle típicos dessa técnica, entre eles `data_inicio_vigencia`, `data_fim_vigencia` e um sinalizador `is_current` (linha vigente) — o que permite reconstruir exatamente qual era o valor de um atributo em qualquer momento do passado.

Existem ainda variações mais avançadas — Tipo 0 (o valor nunca muda), Tipo 3 (guarda o valor anterior em uma coluna extra), e os Tipos 4 a 7 (combinações com tabelas de histórico separadas) — descritas em detalhe no [*Design Tip #152*](https://www.kimballgroup.com/2013/02/design-tip-152-slowly-changing-dimension-types-0-4-5-6-7/) (Ross, M. *Kimball Group*, 2013). Na prática, a grande maioria dos pipelines de dados usa Tipo 1 ou Tipo 2 — e a escolha entre eles é uma decisão de modelagem, não uma regra fixa: depende de quanto histórico o negócio realmente precisa preservar.

### Da Modelagem ao Valor: os Tipos de Analytics

Lembra do último dos 5 V's — o **Valor**? Uma vez que os dados estão bem modelados em fatos e dimensões, a pergunta natural é: que tipo de pergunta de negócio esse modelo é capaz de responder? A literatura de sistemas de apoio à decisão costuma organizar essa resposta em uma escala de complexidade crescente, do mais simples ao mais sofisticado.

No artigo [*"Data, information and analytics as services"*](https://www.sciencedirect.com/science/article/abs/pii/S0167923612001558) (Delen, D., & Demirkan, H. *Decision Support Systems*, 55(1), 359–363, 2013), os autores descrevem essa progressão através de três categorias centrais, cada uma respondendo a uma pergunta diferente sobre o mesmo dado modelado:

- **Analytics Descritiva** ("O que aconteceu?"): resume e organiza dados históricos em relatórios, dashboards e KPIs — é o tipo mais comum, e é exatamente o que uma consulta direta à nossa tabela fato produz (ex.: "quanto vendemos por região no último trimestre?").
- **Analytics Preditiva** ("O que provavelmente vai acontecer?"): usa métodos estatísticos e de machine learning sobre os dados históricos para estimar resultados futuros (ex.: prever a demanda do próximo mês a partir do histórico de vendas).
- **Analytics Prescritiva** ("O que devemos fazer a respeito?"): vai além de prever, recomendando ações concretas, geralmente combinando modelos preditivos com otimização e regras de negócio (ex.: sugerir automaticamente quanto estoque comprar de cada produto).

Na prática de mercado, uma quarta categoria costuma ser inserida entre as duas primeiras — a **Analytics Diagnóstica** ("Por que aconteceu?"): investiga as causas por trás de um resultado descritivo, cruzando dimensões e buscando correlações (ex.: entender por que as vendas caíram em uma região específica). Consultorias e institutos de pesquisa de mercado como Gartner, IBM e SAS popularizaram esse modelo de quatro categorias — descritiva, diagnóstica, preditiva e prescritiva — como uma espécie de "escada de maturidade": cada degrau exige mais sofisticação técnica e organizacional do que o anterior, mas também entrega mais valor para a tomada de decisão.

> 💡 **Dica:** Não é uma questão de "qual tipo de analytics é o melhor" — a maioria das organizações usa os quatro o tempo todo, para perguntas diferentes. E é exatamente o modelo dimensional que vimos nesta seção — fatos, dimensões, grão bem definido, histórico preservado via SCD — que viabiliza os quatro tipos a partir do mesmo conjunto de tabelas.

## Batch vs. Streaming

Vale destacar mais uma distinção importante: a forma como os dados são processados ao longo do tempo.

- **Processamento em lote (Batch)**: os dados são acumulados por um período e processados de uma vez, em intervalos programados. É o caso do conjunto de dados usado nesta aula: extrações mensais completas, particionadas por `dt=AAAA-MM-DD`.
- **Processamento em fluxo contínuo (Streaming)**: os dados são processados evento a evento, assim que chegam, com latência de segundos ou menos. Seria o caso, por exemplo, de processar cada evento (uma venda, uma transação, um clique) no exato momento em que ele acontece.

Por muito tempo, batch e streaming foram tratados como dois mundos com ferramentas e modelos de programação completamente diferentes. O paper [*"The Dataflow Model: A Practical Approach to Balancing Correctness, Latency, and Cost in Massive-Scale, Unbounded, Out-of-Order Data Processing"*](http://www.vldb.org/pvldb/vol8/p1792-Akidau.pdf) (Akidau, T. et al. *Proceedings of the VLDB Endowment*, Vol. 8, No. 12, 2015) propôs um modelo único capaz de descrever os dois cenários com a mesma API, tratando o batch como um caso particular de streaming (um fluxo que, em algum momento, simplesmente para de chegar). Os autores destacam que dados "ilimitados" (que nunca param de chegar) e "fora de ordem" (que podem chegar atrasados, fora da sequência em que aconteceram) são cada vez mais a norma, não a exceção — e que é preciso um jeito de equilibrar corretude, latência e custo ao processá-los. Esse modelo, criado no Google, é a base do **Apache Beam** e influenciou diretamente como ferramentas como o Spark Structured Streaming lidam hoje com tempo de evento e janelas de agregação.

> 📝 **Nota:** Batch e streaming não são mutuamente exclusivos — muitas arquiteturas modernas combinam os dois, processando a maior parte dos dados em lote e alguns fluxos críticos em tempo real. Em sistemas que fazem extrações periódicas completas, como o do exemplo desta aula, o cenário batch costuma ser o mais natural.

## Os Artigos Originais, Reunidos

Ao longo desta aula você viu referências espalhadas pelo texto. Aqui está a lista completa, em ordem cronológica, caso queira ler algum deles na íntegra — todos são de acesso gratuito, com exceção do livro do Inmon:

1. Codd, E. F. (1970). ["A Relational Model of Data for Large Shared Data Banks."](https://github.com/tpn/pdfs/blob/master/A%20Relational%20Model%20of%20Data%20for%20Large%20Shared%20Data%20Banks%20-%20E.F.%20Codd%20(1970).pdf) *Communications of the ACM*, 13(6), 377–387.
2. Inmon, W. H. (início dos anos 1990). *Building the Data Warehouse*. Wiley. (livro, sem link)
3. Reichhardt, T. (1999). ["It's sink or swim as a tidal wave of data approaches."](https://www.nature.com/articles/21044) *Nature*, 399, 517–518.
4. Kimball, R. (2003). ["Fact Tables and Dimension Tables."](https://www.kimballgroup.com/2003/01/fact-tables-and-dimension-tables/) *Kimball Group*.
5. Ghemawat, S., Gobioff, H., & Leung, S.-T. (2003). ["The Google File System."](https://research.google/pubs/the-google-file-system/) *SOSP '03*.
6. Dean, J., & Ghemawat, S. (2004). ["MapReduce: Simplified Data Processing on Large Clusters."](https://research.google/pubs/mapreduce-simplified-data-processing-on-large-clusters/) *OSDI '04*.
7. Kimball, R. (2007). ["Keep to the Grain in Dimensional Modeling."](https://www.kimballgroup.com/2007/07/keep-to-the-grain-in-dimensional-modeling/) *Kimball Group*.
8. Kimball, R. (2008). ["Slowly Changing Dimensions"](https://www.kimballgroup.com/2008/08/slowly-changing-dimensions/) e ["Part 2."](https://www.kimballgroup.com/2008/09/slowly-changing-dimensions-part-2/) *Kimball Group*.
9. Laney, D. (2012). ["Deja VVVu: Gartner's Original 'Volume-Velocity-Variety' Definition of Big Data."](https://community.aiim.org/blogs/doug-laney/2012/08/25/deja-vvvu-gartners-original-volume-velocity-variety-definition-of-big-data) *AIIM Community* (relato sobre a nota de pesquisa original de 2001).
10. Zaharia, M. et al. (2012). ["Resilient Distributed Datasets: A Fault-Tolerant Abstraction for In-Memory Cluster Computing."](https://www.usenix.org/conference/nsdi12/technical-sessions/presentation/zaharia) *NSDI '12*.
11. Delen, D., & Demirkan, H. (2013). ["Data, information and analytics as services."](https://www.sciencedirect.com/science/article/abs/pii/S0167923612001558) *Decision Support Systems*, 55(1), 359–363.
12. Ross, M. (2013). ["Design Tip #152: Slowly Changing Dimension Types 0, 4, 5, 6 and 7."](https://www.kimballgroup.com/2013/02/design-tip-152-slowly-changing-dimension-types-0-4-5-6-7/) *Kimball Group*.
13. Beauchemin, M. (2015). ["Airflow: a workflow management platform."](https://medium.com/airbnb-engineering/airflow-a-workflow-management-platform-46318b977fd8) *Airbnb Engineering & Data Science*.
14. Akidau, T. et al. (2015). ["The Dataflow Model: A Practical Approach to Balancing Correctness, Latency, and Cost in Massive-Scale, Unbounded, Out-of-Order Data Processing."](http://www.vldb.org/pvldb/vol8/p1792-Akidau.pdf) *Proceedings of the VLDB Endowment*, 8(12).
15. Armbrust, M., Ghodsi, A., Xin, R., & Zaharia, M. (2021). ["Lakehouse: A New Generation of Open Platforms that Unify Data Warehousing and Advanced Analytics."](http://cidrdb.org/cidr2021/papers/cidr2021_paper17.pdf) *CIDR '21*.

> 💡 **Dica:** Não é preciso ler nenhum desses papers por completo para entender os conceitos apresentados aqui — mas o resumo (*abstract*) de cada um costuma ser curto e vale muito a pena. É uma ótima forma de treinar o hábito de voltar à fonte primária em vez de confiar apenas em resumos de terceiros (como este notebook!).

## Saiba Mais

- [Documentação do Apache Spark](https://spark.apache.org/docs/latest/)
- [O que é um Data Lakehouse (Delta Lake)](https://delta.io/)
- [Arquitetura Medalhão (Databricks)](https://www.databricks.com/glossary/medallion-architecture)
- [Documentação do Amazon S3](https://docs.aws.amazon.com/s3/)
- [Documentação do Apache Airflow](https://airflow.apache.org/docs/)
- [Kimball Dimensional Modeling Techniques (Kimball Group)](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/kimball-techniques/dimensional-modeling-techniques/)

## Finalizando

Nesta aula você deixou de lado o modismo em torno do termo "Big Data" e ganhou critérios objetivos para reconhecer um problema de fato distribuído: volume, velocidade e variedade ultrapassando os limites de uma única máquina. Você também conheceu as quatro peças centrais do ecossistema — armazenamento distribuído, processamento distribuído, arquitetura em camadas (medalhão) e orquestração —, os fundamentos de modelagem de dados que sustentam a camada Gold (OLTP vs. OLAP, fatos, dimensões, grão e Slowly Changing Dimensions), os quatro tipos de analytics que esse modelo viabiliza (descritiva, diagnóstica, preditiva e prescritiva), e a diferença entre processamento em lote e em fluxo contínuo.

Com esse vocabulário e esses critérios em mãos, você já está preparado para avaliar, de forma criteriosa, quando vale a pena migrar de uma solução single-machine para uma arquitetura distribuída — e para conversar, de igual para igual, sobre como modelar os dados quando chegar a hora de projetar suas próprias tabelas fato e dimensão.

- [Documentação do Apache Spark](https://spark.apache.org/docs/latest/)
- [Arquitetura Medalhão (Databricks)](https://www.databricks.com/glossary/medallion-architecture)
- [Kimball Dimensional Modeling Techniques (Kimball Group)](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/kimball-techniques/dimensional-modeling-techniques/)

Um abraço e até a próxima,

Walter.